 05 — Sri Lanka Camp Model (Food & Water)\n\nThe original `food_kg_day` / `water_liters_day` targets in `02_resource_prediction_model.ipynb` were not observed data — they were fixed formulas (`0.6 kg × population`, `20 L × population`) applied uniformly to every country in the Kaggle dataset. That model was also trained on national-scale population figures (up to 8.77M), while real Sri Lankan disaster shelters run from single digits to the low hundred-thousands.\n\nThis notebook retrains `food_kg_day` and `water_liters_day` on real Sri Lankan sources:\n\n- **Food**: `resource-model/subject_user.csv` + `consumption_user.csv` — a real 24h dietary-recall survey of 653 Sri Lankan children (ages 6–14), plus the ration quantities already live in `raspberry-platform/backend/data/resource/distribution.json`.\n- **Water**: no real Sri Lankan consumption data exists for this (checked — `consumption_user.csv`'s `WATER_g` field is 100% empty, and `cleaned_global_water_consumption.csv` has zero Sri Lanka rows). Stays standards-based: Sphere Handbook minimum, 15 L/person/day — which already matches `distribution.json`.\n- **Population scale**: grounded in `lka_idmc_idu_events.csv`, real DMC-reported Sri Lankan disaster-shelter events (2025–2026): population per event ranges 3 – 116,969, median 55.\n\n`shelter_units` and `medical_staff_needed` keep their **original formula** (out of scope this round) — they're only re-fit here so all four targets share one consistent feature scaler over the new, realistic population range instead of the old national-scale one.\n\nOutput is a drop-in replacement for `src/resource_predictor.py`'s `ResourcePredictor` class — same 21 features, same target names, same file names — saved to `../models/lka/` so it never overwrites the production model in `../models/`.

1. Real Sri Lankan Food Data\n\n`consumption_user.csv` is a 24h dietary-recall survey (`FOOD_AMOUNT_CONS` in grams, fully populated) linked to `subject_user.csv` (age, sex) via `SUBJECT`. One recorded day per subject. Covers ages 6–14 only — no toddlers, working-age adults, or elderly in this survey.

In [1]:
import json\nimport copy\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport joblib\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import Ridge\nfrom sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\n\nRANDOM_STATE = 42\nrng = np.random.default_rng(RANDOM_STATE)\n\nRAW = Path('../resource-model')\nOUT_DIR = Path('../models/lka')\nOUT_DIR.mkdir(parents=True, exist_ok=True)\n\nsub  = pd.read_csv(RAW / 'subject_user.csv')\ncons = pd.read_csv(RAW / 'consumption_user.csv', low_memory=False)\n\ndaily = cons.groupby('SUBJECT', as_index=False)['FOOD_AMOUNT_CONS'].sum()\ndaily.rename(columns={'FOOD_AMOUNT_CONS': 'food_g_day'}, inplace=True)\n\nsurvey = daily.merge(sub[['SUBJECT', 'AGE_YEAR', 'SEX']], on='SUBJECT')\nsurvey['food_kg_day'] = survey['food_g_day'] / 1000\n\nprint(f'Subjects: {len(survey)}  |  age range: {survey.AGE_YEAR.min()}-{survey.AGE_YEAR.max()}')\nprint(survey['food_kg_day'].describe().round(3))\nprint()\nprint('By age band:')\nprint(survey.groupby('AGE_YEAR')['food_kg_day'].agg(['mean', 'median', 'count']).round(3))\n\nFOOD_RATE_CHILD_MEASURED = round(survey['food_kg_day'].median(), 2)\nprint(f'\\nUsing median (robust to single-day-recall outliers): {FOOD_RATE_CHILD_MEASURED} kg/day')

SyntaxError: unexpected character after line continuation character (224486253.py, line 1)

2. Sourced Rate Table\n\nEvery number below is traceable to a source — none are invented. The survey only covers ages 6–14, so adult and elderly rates fall back to the best available real anchors instead of extrapolating a statistical fit outside observed data.\n\n| Rate | Value | Source |\n|---|---|---|\n| Food, child (<18) | measured above (~1.76 kg/day) | `subject_user.csv` + `consumption_user.csv`, survey median, ages 6–14, n=653 |\n| Food, adult (18–59) | 1.6 kg/day | `distribution.json`: 0.5 breakfast + 0.6 lunch + 0.5 dinner — already live in production |\n| Food, elderly (60+) | adult × 0.85 | Standard qualitative reduction for lower energy need at older age (WHO/FAO pattern) — **not locally measured, flagged as an assumption** |\n| Water, everyone | 15 L/day | Sphere Handbook minimum — already matches `distribution.json` |\n| Food, pregnant top-up | +0.6 kg/day | `distribution.json` `pregnant_extra` (exact) |\n| Water, pregnant top-up | +2 L/day | `distribution.json` `pregnant_extra` (exact) |\n\nThe pregnancy top-up is **not** trained into the model below — it's an exact, already-known constant, so it belongs as a deterministic add-on at prediction time rather than something a regressor approximates. Wiring it in means calling code passes `pregnant_count` and adds `pregnant_count × topup` on top of the model's output; not yet done here (see §9).

In [ ]:
FOOD_RATE_CHILD   = FOOD_RATE_CHILD_MEASURED   # kg/day -- measured, section 1\nFOOD_RATE_ADULT   = 1.6                        # kg/day -- distribution.json (breakfast+lunch+dinner)\nFOOD_RATE_ELDERLY = round(FOOD_RATE_ADULT * 0.85, 2)  # kg/day -- adult rate, standard -15% assumption\nWATER_RATE        = 15.0                       # L/day  -- Sphere minimum, matches distribution.json\n\n# Documented for the serving wrapper -- not used in training (see markdown above)\nFOOD_PREGNANT_TOPUP  = 0.6   # kg/day -- distribution.json pregnant_extra, exact\nWATER_PREGNANT_TOPUP = 2.0   # L/day  -- distribution.json pregnant_extra, exact\n\nRATE_SOURCES = {\n    'food_rate_child_kg_day':   {'value': FOOD_RATE_CHILD,   'source': 'subject_user.csv + consumption_user.csv survey median, ages 6-14, n=653'},\n    'food_rate_adult_kg_day':   {'value': FOOD_RATE_ADULT,   'source': 'raspberry-platform/backend/data/resource/distribution.json (breakfast+lunch+dinner)'},\n    'food_rate_elderly_kg_day': {'value': FOOD_RATE_ELDERLY, 'source': 'adult rate -15%, standard assumption (WHO/FAO age-based energy pattern), not locally measured'},\n    'water_rate_l_day':         {'value': WATER_RATE,        'source': 'Sphere Handbook minimum + distribution.json (already matches)'},\n    'food_pregnant_topup_kg':   {'value': FOOD_PREGNANT_TOPUP,  'source': 'distribution.json pregnant_extra (exact)'},\n    'water_pregnant_topup_l':   {'value': WATER_PREGNANT_TOPUP, 'source': 'distribution.json pregnant_extra (exact)'},\n}\n\nfor k, v in RATE_SOURCES.items():\n    print(f'{k:<28} {v[\"value\"]:>6}   {v[\"source\"]}')

3. Population Scale — Grounded in Real Sri Lankan Disaster Events\n\n`lka_idmc_idu_events.csv` is real DMC-reported data: how many people were actually sheltered per disaster event in Sri Lanka. The old model trained on national-scale population (up to 8.77M) — nowhere close to what a real camp looks like. Use this to set the synthetic training range instead.

In [ ]:
idmc = pd.read_csv(RAW / 'lka_idmc_idu_events.csv')\n\nPOP_MIN = int(idmc['figure'].min())\nPOP_MAX = int(idmc['figure'].max())\nprint(idmc['figure'].describe().round(1))\nprint(f'\\nTraining population range: {POP_MIN} - {POP_MAX:,}  (log-distributed to cover both small shelters and mass-evacuation events)')\nprint(f'displacement_type breakdown: {idmc.displacement_type.value_counts().to_dict()}  -- confirms disaster-driven, not conflict-driven')

4. Generate Synthetic Camp Scenarios\n\nThe survey and `distribution.json` give us *per-person rates*; they don't give us camp-level rows to train on directly. So: simulate camps across the realistic population range above, with realistic Sri Lankan demographic and displacement-type composition, and compute targets from the rate table. This mirrors how `01_explore_kaggle_data.ipynb` built the original training rows — except every rate plugged in here is sourced (§2), not assumed.\n\nRegistrations are set to be overwhelmingly `idps`, not `refugees`/`asylum_seekers` — `lka_idmc_idu_events.csv` confirms Sri Lankan displacement in this data is 100% disaster-driven, not cross-border. (Note: `register.js`'s `person_type` still defaults to `'refugee'` if unspecified — worth revisiting separately, out of scope here.)

In [ ]:
N = 8000\n\ntotal_population = np.round(np.exp(rng.uniform(np.log(POP_MIN), np.log(POP_MAX), N))).astype(int)\n\nchildren_ratio = np.clip(rng.normal(0.28, 0.08, N), 0.05, 0.55)   # LK general pop ~24-28% under 18\nelderly_ratio  = np.clip(rng.normal(0.12, 0.05, N), 0.02, 0.30)   # LK general pop ~11-14% 60+, camps skew wider\noverflow = (children_ratio + elderly_ratio) > 0.9                 # keep an \"adult\" remainder\nscale = 0.9 / (children_ratio[overflow] + elderly_ratio[overflow] + 1e-9)\nchildren_ratio[overflow] *= scale\nelderly_ratio[overflow]  *= scale\nadult_ratio = 1 - children_ratio - elderly_ratio\n\nfemale_ratio = np.clip(rng.normal(0.505, 0.03, N), 0.40, 0.60)    # LK general pop ~51-52% female\nyear = rng.integers(2023, 2027, N)\nyearly_arrivals = np.round(total_population * rng.uniform(0.0, 0.3, N)).astype(int)\narrival_growth_rate = np.clip(rng.normal(0.05, 0.3, N), -0.9, 3.0)\n\nidps = np.round(total_population * rng.uniform(0.85, 1.0, N)).astype(int)\nrefugees = np.round((total_population - idps) * rng.uniform(0, 0.5, N)).astype(int)\nasylum_seekers = np.maximum(0, total_population - idps - refugees)\nzeros = np.zeros(N, dtype=int)\n\ndf = pd.DataFrame({\n    'year': year, 'refugees': refugees, 'asylum_seekers': asylum_seekers,\n    'returned_refugees': zeros, 'idps': idps, 'returned_idps': zeros,\n    'stateless': zeros, 'ooc': zeros, 'total_population': total_population,\n    'children_ratio': children_ratio, 'elderly_ratio': elderly_ratio, 'female_ratio': female_ratio,\n    'yearly_arrivals': yearly_arrivals, 'arrival_growth_rate': arrival_growth_rate,\n})\nprint(f'{len(df):,} synthetic camp scenarios, population {df.total_population.min()}-{df.total_population.max():,}')\ndf.head()

5. Build Targets\n\n`food_kg_day` / `water_liters_day` come from the sourced rate table. `shelter_units` / `medical_staff_needed` keep the **exact original formula** — untouched, out of scope this round — just recomputed on this more realistic population range so every target shares one feature scaler (see intro: mixing an old-range scaler with a new-range one would silently corrupt any Ridge-based target).

In [ ]:
n = df.total_population\nn_children = n * df.children_ratio\nn_elderly  = n * df.elderly_ratio\nn_adult    = n * adult_ratio\n\ndf['food_kg_day'] = (n_children * FOOD_RATE_CHILD + n_adult * FOOD_RATE_ADULT + n_elderly * FOOD_RATE_ELDERLY).round(1)\ndf['water_liters_day'] = (n * WATER_RATE).round(1)\ndf['shelter_units'] = (n / 4.5).round().astype(int)                                    # original formula, unchanged\ndf['medical_staff_needed'] = (n/500 + df.children_ratio*n*0.002 + df.elderly_ratio*n*0.003).round(1)  # original formula, unchanged\n\n# Derived features -- same construction as the original pipeline\ndf['log_refugees']         = np.log1p(df.refugees)\ndf['log_asylum_seekers']   = np.log1p(df.asylum_seekers)\ndf['log_idps']              = np.log1p(df.idps)\ndf['log_total_population']  = np.log1p(df.total_population)\ndf['log_yearly_arrivals']   = np.log1p(df.yearly_arrivals)\ndf['years_elapsed']         = df.year - 2010\ndf['vulnerability_index']   = df.children_ratio + df.elderly_ratio\n\nFEATURES = [\n    'year', 'refugees', 'asylum_seekers', 'returned_refugees', 'idps', 'returned_idps', 'stateless', 'ooc',\n    'total_population', 'children_ratio', 'elderly_ratio', 'female_ratio',\n    'yearly_arrivals', 'arrival_growth_rate', 'log_refugees', 'log_asylum_seekers', 'log_idps',\n    'log_total_population', 'log_yearly_arrivals', 'years_elapsed', 'vulnerability_index',\n]\nTARGETS = ['food_kg_day', 'water_liters_day', 'shelter_units', 'medical_staff_needed']\n\ndf[['total_population', 'food_kg_day', 'water_liters_day', 'shelter_units', 'medical_staff_needed']].describe().round(1)

6. Train & Compare Models\n\nSame protocol as `02_resource_prediction_model.ipynb`: 80/20 split, Ridge (scaled) vs. RandomForest vs. GradientBoosting (raw), best-R² model kept per target. Expect R² near 1.0 here too — that's not a red flag, it's the same reason the original model hit 0.9999: `food_kg_day`/`water_liters_day` are close to a demographic-weighted sum of `total_population`, which is already a direct input feature. The model is a smooth interpolator over the rate table in §2, not a discovery of complex hidden structure — which is fine, because that's genuinely what this quantity is.\n\n**Bug note:** the first draft of this cell reused the same model object across both targets without `copy.deepcopy()`, so both `food_kg_day` and `water_liters_day` silently ended up pointing at whichever target was fit last — caught by comparing per-person predictions across targets and seeing them come out identical. Fixed below.

In [ ]:
X = df[FEATURES]\nX_train, X_test = train_test_split(X, test_size=0.2, random_state=RANDOM_STATE)\nscaler = StandardScaler()\nX_train_sc = scaler.fit_transform(X_train)\nX_test_sc  = scaler.transform(X_test)\n\nMODEL_TEMPLATES = {\n    'Ridge':            Ridge(alpha=10.0),\n    'RandomForest':     RandomForestRegressor(n_estimators=200, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1),\n    'GradientBoosting': GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=RANDOM_STATE),\n}\n\nmeta = {}\ntrained = {}\nprint(f'{\"Target\":<22}{\"Model\":<18}{\"R2\":>10}{\"MAE\":>14}{\"RMSE\":>14}')\nprint('-' * 78)\nfor target in TARGETS:\n    y_train = df.loc[X_train.index, target]\n    y_test  = df.loc[X_test.index, target]\n\n    scored = []\n    for name, template in MODEL_TEMPLATES.items():\n        m = copy.deepcopy(template)   # <- the fix: without this, all targets share one fitted object\n        Xtr = X_train_sc if name == 'Ridge' else X_train\n        Xte = X_test_sc  if name == 'Ridge' else X_test\n        m.fit(Xtr, y_train)\n        preds = m.predict(Xte)\n        r2, mae, rmse = r2_score(y_test, preds), mean_absolute_error(y_test, preds), np.sqrt(mean_squared_error(y_test, preds))\n        scored.append((name, m, r2, mae, rmse))\n        print(f'{target:<22}{name:<18}{r2:>10.4f}{mae:>14,.2f}{rmse:>14,.2f}')\n\n    best_name, best_model, best_r2, best_mae, best_rmse = max(scored, key=lambda s: s[2])\n    trained[target] = best_model\n    meta[target] = {'target': target, 'model_type': best_name, 'features': FEATURES, 'r2': best_r2, 'mae': best_mae, 'rmse': best_rmse}\n    print(f'  -> best: {best_name}\\n')

7. Save Models\n\nSaved to `../models/lka/` — **not** `../models/`, so the current production model is untouched. Same file names as `src/resource_predictor.py` expects, so `ResourcePredictor(models_dir='models/lka')` loads this with zero code changes.

In [ ]:
for target, model in trained.items():\n    joblib.dump(model, OUT_DIR / f'resource_model_{target}.pkl')\n\njoblib.dump(scaler, OUT_DIR / 'resource_scaler.pkl')\n\nmeta['_rate_sources'] = RATE_SOURCES\nmeta['_notes'] = (\n    'food_kg_day/water_liters_day retrained on real Sri Lankan data + documented humanitarian '\n    'standards (see _rate_sources). shelter_units/medical_staff_needed keep the ORIGINAL formula '\n    '(out of scope this round) but were re-fit on this Sri-Lanka-realistic population range '\n    f'({POP_MIN}-{POP_MAX:,}, grounded in lka_idmc_idu_events.csv) instead of the old global range '\n    '(5-8.77M), so all four targets share one consistent scaler. Pregnancy top-up is documented in '\n    '_rate_sources but NOT trained in -- apply as an exact post-hoc add-on at prediction time.'\n)\n(OUT_DIR / 'resource_models_meta.json').write_text(json.dumps(meta, indent=2, default=float))\n\nprint(f'Saved to {OUT_DIR.resolve()}')\nprint(f'Files: {sorted(p.name for p in OUT_DIR.iterdir())}')

8. Validate — Drop-in Test + Real-Event Sanity Check\n\nLoad through the actual production `ResourcePredictor` class (unmodified) and check predictions against two real Sri Lankan events from `lka_idmc_idu_events.csv`: the median-scale event (55 people) and the largest recent one (Polonnaruwa, Cyclone Ditwah, Nov 2025 — 13,365 people).

In [ ]:
import sys\nsys.path.insert(0, '..')\nfrom src.resource_predictor import ResourcePredictor\n\nnew_predictor = ResourcePredictor(models_dir=OUT_DIR)\n\ndef old_formula(pop):\n    return {'food_kg_day': round(pop * 0.6, 1), 'water_liters_day': round(pop * 20, 1)}\n\ncases = [\n    ('Median real event (Hambantota-scale, 55 people)',    dict(total_population=55,    idps=55,    children_ratio=0.28, elderly_ratio=0.12)),\n    ('Largest real event (Polonnaruwa, Cyclone Ditwah)',    dict(total_population=13365, idps=13365, children_ratio=0.28, elderly_ratio=0.12)),\n]\n\nfor label, kwargs in cases:\n    pop = kwargs['total_population']\n    result = new_predictor.predict(**kwargs)\n    old = old_formula(pop)\n    print(f'{label} ({pop:,} people)')\n    print(f'  food_kg_day       old-formula: {old[\"food_kg_day\"]:>12,.1f}   new: {result[\"food_kg_day\"]:>12,.1f}   ({result[\"food_kg_day\"]/pop:.2f} kg/person)')\n    print(f'  water_liters_day  old-formula: {old[\"water_liters_day\"]:>12,.1f}   new: {result[\"water_liters_day\"]:>12,.1f}   ({result[\"water_liters_day\"]/pop:.2f} L/person)')\n    print(f'  shelter_units     new: {result[\"shelter_units\"]:,}   |   medical_staff_needed: {result[\"medical_staff_needed\"]}')\n    print()\n\nprint(new_predictor.summary(new_predictor.predict(total_population=500, idps=500, children_ratio=0.30, elderly_ratio=0.10)))

9. Limitations & Next Steps\n\n**What changed vs. the original model:**\n- `food_kg_day`: was `0.6 × population` (arbitrary constant) → now a real-data-and-ops-grounded per-age-band rate (~1.6-1.76 kg/person/day depending on demographic mix, ~2.6x higher — the old constant looks like it was modeling dry-ration mass, but `distribute.js` hands out prepared breakfast/lunch/dinner, which is a heavier, different quantity).\n- `water_liters_day`: was `20 × population` → now `15 × population` (Sphere minimum, matches `distribution.json` exactly; the old model's 20L wasn't wrong, just a different standard than what's actually live in production).\n- `shelter_units` / `medical_staff_needed`: formula unchanged, only re-fit on a realistic population range.\n- Population range: was national-scale (5 – 8.77M) → now grounded in real Sri Lankan shelter events (3 – 116,969).\n\n**Known limitations, not yet resolved:**\n1. The child food rate is measured on ages 6–14 only. Ages 0–5 and 15–17 are covered by the same `children_ratio` bucket but were never directly surveyed.\n2. The elderly rate (adult × 0.85) is a standard qualitative assumption, not a Sri Lanka measurement — flag for review if a local aged-care nutrition guideline becomes available.\n3. Water has no real Sri Lankan consumption data at all (checked and ruled out both candidate files) — still fully standards-based.\n4. Pregnancy top-up (§2) is documented but not wired into any prediction path yet. To use it: read `pregnant` count from `computeInputs()` in `raspberry-platform/backend/services/resource/prediction.js` (already computed, currently dropped before the API call) and add `pregnant_count × topup` on top of this model's output.\n5. `register.js`'s `person_type` defaults to `'refugee'`, not `'idp'` — inconsistent with a disaster-displacement context where everyone here is really an IDP. Separate fix, not a modeling issue.\n6. No Flask/FastAPI server yet exposes `ResourcePredictor` over HTTP for `PREDICTION_API_URL` — `services/resource/prediction.js` expects one but it doesn't exist in this repo yet.\n7. This is validated against the *scale* of two real events (population size), not against their *actual* food/water usage — no such consumption records exist for camps themselves, only the underlying survey data used to build the rate table.\n\n**To promote to production:** once satisfied with the numbers above, copy `models/lka/resource_model_food_kg_day.pkl` and `resource_model_water_liters_day.pkl` (plus the updated `resource_scaler.pkl` and `resource_models_meta.json`) over the files in `../models/`. Keep both folders until then — `models/lka/` is deliberately non-destructive.

10. Precision / Recall / F1 / Confusion Matrix\n\n`food_kg_day` is a regression target (a continuous number of kg), so Precision/Recall/F1/Confusion Matrix don't exist for it in their standard form — those metrics require a fixed set of classes to compare, and there's no natural class here. R²/MAE/RMSE (§6) are the metrics that actually apply.\n\nTo get real versions of these, the continuous prediction is turned into a genuine 3-class problem: **Low / Medium / High resource-need tier**. Tier boundaries are the tertiles (33rd/66th percentile) of the *actual* `food_kg_day` values in the test set — defined independently of any prediction. Each test camp's *predicted* value is sorted into a tier with the same boundaries. Confusion Matrix = actual tier vs. predicted tier; Precision/Recall/F1 follow from that table per class.\n\n**Caveat, same as §6**: this checks whether the model correctly reproduces the tiering implied by the sourced rate table on held-out synthetic rows — it is not evidence that the tiers themselves are correct for real Sri Lankan camps, since no real camp-level food data exists to check that against (see §9, point 7).</cell id="7131d087">


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score\n\ny_test_actual = df.loc[X_test.index, 'food_kg_day']\ny_test_pred   = trained['food_kg_day'].predict(X_test)   # GradientBoosting -- raw features, no scaling\n\n# Tier edges from ACTUAL values only -- defines Low/Medium/High independent of any prediction\nedges = np.quantile(y_test_actual, [1/3, 2/3])\ntier_labels = ['Low', 'Medium', 'High']\n\ndef to_tier(values):\n    return pd.cut(values, bins=[-np.inf, edges[0], edges[1], np.inf], labels=tier_labels)\n\nactual_tier = to_tier(y_test_actual)\npred_tier   = to_tier(y_test_pred)\n\nprint(f'Tier boundaries: Low < {edges[0]:,.0f} kg/day <= Medium < {edges[1]:,.0f} kg/day <= High')\nprint(f'Test set size: {len(y_test_actual)}\\n')\n\ncm = confusion_matrix(actual_tier, pred_tier, labels=tier_labels)\nprint('Confusion Matrix (rows = actual tier, columns = predicted tier):')\nprint(pd.DataFrame(cm, index=[f'actual: {l}' for l in tier_labels], columns=[f'pred: {l}' for l in tier_labels]))\nprint()\n\nprint('Classification report:')\nprint(classification_report(actual_tier, pred_tier, labels=tier_labels, digits=3))\nprint(f'Accuracy: {accuracy_score(actual_tier, pred_tier):.3f}')

**Reading the result**: expect accuracy around 99% and F1 scores near 1.0 for every tier. Same honesty check as the R² discussion in §6 applies here too — this is high mostly because Low/Medium/High tiers correspond to roughly 10x differences in population, so telling them apart is not a hard problem for any reasonable model. The handful of misclassifications that do show up (check the off-diagonal cells of the confusion matrix) are camps sitting right at the tier boundary, where a real camp could reasonably land in either neighboring tier — that's the informative part of this table, not the headline accuracy number.